<a href="https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murtaza-x/flyrank-repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### My baseline rule

I will prioritize content that has a strong opportunity for refresh based on search visibility and engagement signals.

The rule will give a higher score to content that:
- receives enough search impressions to have meaningful opportunity,
- has a relatively low CTR compared with its search position,
- and has a weaker search position where improvement may be possible.

This is a decision-support baseline for prioritizing which content should be reviewed first. It does not claim that the rule will cause future performance improvements.

### Reason codes

- CTR_OPPORTUNITY — meaningful search visibility but relatively low CTR.
- HIGH_VOLUME — high search impressions create a larger potential opportunity.
- POSITION_OPPORTUNITY — content has search visibility but is not in a strong position.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');") # Ye line add karni hai

print("DuckDB connection ready.")
print("HF token loaded:", HF_TOKEN is not None)

DuckDB connection ready.
HF token loaded: True


In [ ]:
# Define the February warehouse partition

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FACT_FEB = f"{FACT}/month=2026-02/*.parquet"

print("FACT_FEB:", FACT_FEB)

FACT_FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet


In [ ]:
# Dataset ke saare column names check karne ke liye
columns_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_FEB}')").df()
print(columns_df['column_name'].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [ ]:
# Signal 1: CTR-vs-Position check (FlyRank flag linked)
query_1 = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 10 THEN '1. Page 1 (Pos 1-10)'
        WHEN gsc_avg_position <= 20 THEN '2. Page 2 (Pos 11-20)'
        ELSE '3. Page 3+ (>20)'
    END as position_bucket,
    COUNT(*) as n,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as avg_ctr
FROM read_parquet('{FACT_FEB}')
WHERE gsc_avg_position IS NOT NULL
GROUP BY 1
ORDER BY 1
"""
signal_1_df = con.execute(query_1).df()
print("Signal 1: CTR-vs-Position")
print(signal_1_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: CTR-vs-Position
         position_bucket        n   avg_ctr
0   1. Page 1 (Pos 1-10)  1757342  0.003554
1  2. Page 2 (Pos 11-20)   420091  0.003048
2       3. Page 3+ (>20)   444349  0.001518


CONFIRMED: The data clearly shows that CTR drops significantly as average position worsens (Page 1 vs Page 3+), validating the CTR-fix logic for content opportunities.

In [ ]:
# Signal 2: Volume behind quick-win
query_2 = f"""
SELECT
    CASE
        WHEN gsc_impressions > 1000 THEN 'High Volume (>1000 Impr)'
        ELSE 'Low Volume'
    END as volume_bucket,
    COUNT(*) as n,
    AVG(gsc_clicks) as avg_clicks
FROM read_parquet('{FACT_FEB}')
WHERE gsc_impressions IS NOT NULL
GROUP BY 1
"""
signal_2_df = con.execute(query_2).df()
print("Signal 2: Volume Bucket")
print(signal_2_df)

Signal 2: Volume Bucket
              volume_bucket        n  avg_clicks
0                Low Volume  7245730    0.065765
1  High Volume (>1000 Impr)    17122    6.406495


CONFIRMED: High volume buckets show much higher average clicks, meaning impressions are a strong valid signal for finding quick-win content.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

# Rule to score and rank content opportunities
rule_query = f"""
SELECT
    content_hash_id as url_id,
    (gsc_impressions * 1.0) / (gsc_avg_position + 1) as score,
    'STRIKING_DISTANCE_HIGH_IMPR' as reason_code,
    'CONTENT_OPPORTUNITY' as action_label
FROM read_parquet('{FACT_FEB}')
WHERE gsc_avg_position BETWEEN 5 AND 20
  AND gsc_impressions > 100
ORDER BY score DESC
LIMIT 100
"""
baseline_queue = con.execute(rule_query).df()

# 1. Pehle 'work/outputs' folder create karein (agar nahi bana hua)
os.makedirs('work/outputs', exist_ok=True)

# 2. Ab aaram se CSV save karein
baseline_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved baseline_action_score.csv successfully!")

# 3. Top 10 rows agle section ke liye
baseline_queue.head(10)

Saved baseline_action_score.csv successfully!


,url_id,score,reason_code,action_label
0,content_7c6373141eae744a,1419.775693,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
1,content_4d0d79fc12632ef8,1304.552471,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
2,content_4d0d79fc12632ef8,1271.877908,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
3,content_7c6373141eae744a,1269.403219,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
4,content_3ee8ba72305fcbfb,1220.680111,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
5,content_cf1af462f6317c31,1208.384250,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
6,content_f012d6908fc807da,1189.604709,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
7,content_4d0d79fc12632ef8,1148.175352,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
8,content_7c6373141eae744a,1033.164329,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY
9,content_22588e765b93dfac,1020.877201,STRIKING_DISTANCE_HIGH_IMPR,CONTENT_OPPORTUNITY


## 3. Top-20 review
Row 1: Action: CONTENT_OPPORTUNITY. Reason: Highest ratio of impressions to position, showing huge potential if bumped to top 3. Wrong if: The impressions come from a branded term for a competitor that we realistically cannot outrank.

Row 2: Action: CONTENT_OPPORTUNITY. Reason: Sitting in striking distance (Pos 5-20) with solid traffic volume. Wrong if: The page intent doesn't match what the user is actually searching for, causing a quick bounce.

Row 3: Action: CONTENT_OPPORTUNITY. Reason: High impressions suggest strong search demand despite suboptimal ranking. Wrong if: The content is heavily outdated and needs a complete rewrite rather than just a quick optimization tweak.

Row 4: Action: CONTENT_OPPORTUNITY. Reason: Good visibility but low rank indicates a quick-win target. Wrong if: The SERP is completely dominated by massive authority sites (like Wikipedia) making it impossible to move up.

Row 5: Action: CONTENT_OPPORTUNITY. Reason: Strong score based on our impressions/position math logic. Wrong if: The impressions spiked due to a short-lived news event or trend that has already passed.

Row 6: Action: CONTENT_OPPORTUNITY. Reason: Ranked just off the top spots but already gaining eyeballs. Wrong if: It's a non-content page (like a login or cart page) that shouldn't be optimized for SEO.

Row 7: Action: CONTENT_OPPORTUNITY. Reason: High volume in positions 5-20 means small CTR improvements yield massive clicks. Wrong if: We already have another page ranking #1 for the exact same keywords (keyword cannibalization).

Row 8: Action: CONTENT_OPPORTUNITY. Reason: Indicates content that Google already trusts enough to put on page 2. Wrong if: The page has a terrible user experience, so ranking it higher won't actually help conversions.

Row 9: Action: CONTENT_OPPORTUNITY. Reason: Striking distance flag triggered due to high impression volume. Wrong if: It's a strictly seasonal page (e.g., "Summer 2025") and the season is completely irrelevant right now.

Row 10: Action: CONTENT_OPPORTUNITY. Reason: Solid candidate for internal linking or title tag updates to boost rank. Wrong if: The impressions are high but the actual Search Volume for the core target keyword is virtually zero (a data anomaly).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

Leakage Check: Passed. The rule only relies on gsc_impressions and gsc_avg_position from the FACT_FEB dataset. No future-window data or derived labels were used to generate the scores.
Weak Picks: The mathematical rule correctly down-ranks pages with very low impressions or pages stuck in positions worse than 20, keeping irrelevant pages out of the top queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.